In [ ]:
# Install Python packages
%pip install anthropic python-dotenv

# Load env variable
from dotenv import load_dotenv
import os

load_dotenv()

# Create an API Client
from anthropic import Anthropic

client = Anthropic(
    api_key=os.getenv("ANTHROPIC_API_KEY"),
    base_url=os.getenv("ANTHROPIC_BASE_URI")
)
model = "claude-sonnet-4-6"

# Making a Helper Fxn
def add_user_message(messages, text):
    user_message = {"role":"user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role":"assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system = None, temperature = 1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature
    }
    
    if system:
        params["system"] = system

    if stop_sequences:
        params["stop_sequences"]=stop_sequences

    message = client.messages.create(**params)
    return message.content[0].text

In [ ]:
import json
import re

def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
        "format": "json" or "python" or "regex",
        "solution_criteria": "Key Criteria for evaluating the solution"
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""
    messages = []

    add_user_message(messages, prompt)
    
    text = chat(messages)
    match = re.search(r'\[.*\]', text, re.DOTALL)
    if match:
        json_string = match.group(0)
    else:
        json_string = text  # Fallback if Claude returned clean JSON directly
    
    clean_string = json.loads(json_string)
    
    return clean_string
    

In [ ]:
dataset = generate_dataset()

with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)
    
dataset

In [ ]:
def run_prompts(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""Please Solve the following Task
    
    {test_case["task"]}
    
    * Respond only with Python, Json, or plain Regex
    * Do not add any comments or commentary or explaination
    """
    messages = []
    
    add_user_message(messages, prompt)
    output = chat(messages)
    
    return output
    

In [ ]:
def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Criteria you should use to evaluate the function:
<criteria>
{test_case["solution_criteria"]}
</criteria>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
    """

    messages = []
    add_user_message(messages, eval_prompt)
    text = chat(messages)
    
    # Added regex extraction fallback for JSON markdown blocks
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if match:
        json_string = match.group(0)
    else:
        json_string = text
    eval_text = json.loads(json_string)
    return eval_text
    

In [ ]:
import re
import ast

def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0
    
def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0
    
def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0

def grade_syntax(response, test_case):
    format = test_case["format"]
    print(f"Testing format: {format}")  # 👈 Temporary debug line
    
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    elif format == "regex":
        return validate_regex(response)
    else:
        return 0


In [ ]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompts(test_case)
    
    # TODO - Grading
    model_grade = grade_by_model(test_case, output)
    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]
    
    try:
        import json
        parsed_output = json.loads(output)
        code_to_check = parsed_output.get("output", output) 
    except Exception:
        # Fallback to raw output if it isn't valid JSON
        code_to_check = output

    # 3. Pass the clean snippet into the syntax checker
    syntax_score = grade_syntax(code_to_check, test_case)
    
    score = (model_score + syntax_score) / 2
    
    print("Syntax Score:")
    print(syntax_score)
    
    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning
    }

In [ ]:
import statistics

def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
        
    average_score = statistics.mean([result["score"] for result in results])
    print(f"Average Score: {average_score}")
    return results


In [ ]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)
    
results = run_eval(dataset)

In [ ]:
print(json.dumps(results, indent=2))